In [ ]:
from utilities import initialize_with_module

await initialize_with_module('AITW-wood_stiffness_homogenization/1.0.0-gfbf-2025b', '2025.06')

In [ ]:
import utilities
from common import WOOD_SITFFNESS_HOMOG_DEFAULTS
from wood_stiffness_homogenization.cli.run import run

widget_eval, eval_event = utilities.click_cli_widget(
    run,
    override_defaults=WOOD_SITFFNESS_HOMOG_DEFAULTS,
    title='Multiscale Wood Stiffness',
    description_text=(
        "This code computes the multiscale stiffness of native, polymer-infiltrated, and delignified + infiltrated "
        "wood. It works from the cell-wall constituents up to clearwood. It is a procedural port of `02_stiffness/"
        "hom_TranspWood_model.m`, with a lumped cell wall: the middle lamella (ML) and S2 layer are not separated."
    ),
    button_description='Evaluate',
    button_running_text='Evaluating...',
    output_dir_pararm='output',
)

display(widget_eval)

In [ ]:
# Rewrite of the above cell with autodetection of which folders contains the images
import re
import os
import pandas as pd
import ipywidgets as wdg

rgx_output = re.compile(r"(\S+).csv")

title = wdg.HTML(value="<h2>Wood Stiffness Results</h2>")

def find_outputs(base_path="../results/stiff_homog/"):
    outputs = []
    if not os.path.exists(base_path):
        return outputs
    for path in os.listdir(base_path):
        if not os.path.isfile(os.path.join(base_path, path)):
            continue
        match = rgx_output.match(path)
        if not match:
            continue
        label, = match.groups()
        outputs.append((label, os.path.join(base_path, path)))

    return outputs

# Start with a choice widget to select which output to visualize
output_selector = wdg.Dropdown(
    options=[],
    description='Select Output:',
    disabled=False,
    style={'description_width': '150px'},
)
output_selector.layout.width = '500px'

def update_output_options(*args):
    """Update the options of the output selector based on the available outputs in the results folder."""
    outputs = find_outputs()
    options = []
    for label, path in sorted(outputs):
        options.append((f"{label}", path))
    output_selector.options = options
    if output_selector.value is None:
        output_selector.value = options[0][1] if options else None

# Observe changes in the common event to update the output options whenever a new microstructure is generated
try:
    eval_event.observe(update_output_options, names='value')
except Exception as e:
    print("common-event not defined.")
    print("Re-launch this cell after executing the previous one to enable automatic updates of the dropdown choices.")


output_area = wdg.Output()


def display_csv_output(*args):
    """Display the selected image based on the selected output, toggle state, and slider value."""
    selected_output = output_selector.value
    if not selected_output:
        return

    with open(os.path.join(selected_output), 'r') as f:
        data = pd.read_csv(f)

    with output_area:
        output_area.clear_output(wait=True)
        display(data)

# Observe changes in the toggle and slider to update the displayed image
output_selector.observe(display_csv_output, names='value')

# Initial update of the output options and image options
update_output_options()

display(title, output_selector, output_area)